In [1]:
%pip install -q earthengine-api
import ee
ee.Authenticate()
ee.Initialize(project="kinetic-anvil-457906-e8")
print("Conectado a Earth Engine:", ee.String("ok").getInfo())

Note: you may need to restart the kernel to use updated packages.


Enter verification code:  4/1AUn6Hd9myLkaM8NRHjFrIp4ivZS41f9VW_ScYnOWODH3EX4xwa3yZFH-msY



Successfully saved authorization token.
Conectado a Earth Engine: ok


In [2]:
import pandas as pd

SITES = {"Ipiales":(0.83,-77.64),"Nairobi":(-1.29,36.82),"Sorriso":(-12.54,-55.71),"Des Moines":(41.60,-93.61),
 "Ludhiana":(30.90,75.85),"Niamey":(13.51,2.11),"Can Tho":(10.03,105.78),"Mendoza":(-32.89,-68.84),"Fallon":(39.47,-118.78),
 "Ventaquemada":(5.37,-73.52),"Huancayo":(-12.07,-75.21),"Puno":(-15.84,-70.02)}
pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo,la]),{"site":n}) for n,(la,lo) in SITES.items()])

START, END = "2020-01-01", "2025-12-31"   # <- cambia START a "2001-01-01" para la serie completa de NDVI

def monthly_ts(coll, band, scale, factor=1.0):
    coll = coll.select(band).filterDate(START, END)
    n = ee.Date(END).difference(ee.Date(START), "month").toInt()
    months = ee.List.sequence(0, n.subtract(1))
    def mk(k):
        s = ee.Date(START).advance(k, "month")
        img = coll.filterDate(s, s.advance(1, "month")).mean().multiply(factor)
        return img.set("month", s.format("YYYY-MM"))
    imgs = ee.ImageCollection(months.map(mk))
    def samp(img):
        return img.reduceRegions(pts, ee.Reducer.mean(), scale).map(lambda f: f.set("month", img.get("month")))
    fc = imgs.map(samp).flatten()
    rows = fc.getInfo()["features"]
    return pd.DataFrame([{"site":r["properties"].get("site"),"month":r["properties"].get("month"),
                          band:r["properties"].get("mean")} for r in rows])

ndvi = monthly_ts(ee.ImageCollection("MODIS/061/MOD13Q1"), "NDVI", 250, factor=0.0001)
smap_s = monthly_ts(ee.ImageCollection("NASA/SMAP/SPL4SMGP/008"), "sm_surface", 11000)
smap_r = monthly_ts(ee.ImageCollection("NASA/SMAP/SPL4SMGP/008"), "sm_rootzone", 11000)

gee = ndvi.merge(smap_s, how="outer", on=["site","month"]).merge(smap_r, how="outer", on=["site","month"])
gee.to_csv("../data/gee_monthly.csv", index=False)
print("filas:", len(gee), "| sitios:", gee.site.nunique(), "| meses:", gee.month.nunique())
print(gee[gee.site=="Ventaquemada"].head(6).to_string(index=False))

filas: 852 | sitios: 12 | meses: 71
        site   month    NDVI  sm_surface  sm_rootzone
Ventaquemada 2020-01 0.70480    0.226742     0.229164
Ventaquemada 2020-02 0.58205    0.207265     0.215997
Ventaquemada 2020-03 0.57135    0.224797     0.218231
Ventaquemada 2020-04 0.62035    0.242125     0.237479
Ventaquemada 2020-05 0.63245    0.255490     0.242248
Ventaquemada 2020-06 0.56635    0.269427     0.254169
